In [1]:
!pip install scikit-learn -q

In [2]:
import os
import pandas as pd

from google.colab import drive

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_FOLDER="/content/drive/MyDrive/CBR-PHI"

DATA_PATH=os.path.join(
    PROJECT_FOLDER,
    "data",
    "processed",
    "cases.csv"
)

df=pd.read_csv(DATA_PATH)

df.head()

,case_id,nomor_perkara,tahun,jenis_perkara,pemohon,termohon,amar_putusan,isi_putusan
0,1,k pdt sus phi g e n demi nkeadilan berdasarkan...,2022.0,Perselisihan Hubungan Industrial,NaN,an kasasi dan peninjauan kembali e menghukum t...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
1,2,k pdt sus phi e n n demi keadilan berdasarkan ...,2021.0,Perselisihan Hubungan Industrial,NaN,l m b lisa marini warga negara indonesia dahul...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
2,3,k pdt sus phi e n n demi keadilan berdasarkan ...,2021.0,Perselisihan Hubungan Industrial,NaN,ane banding ataupun kasasi a r a i s m ghalama...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
3,4,k pdt sus phi g e n demi nkeadilan berdasarkan...,NaN,Perselisihan Hubungan Industrial,NaN,dengan saksama a diajukan dalam tenggang rwakt...,Ditolak,a i s e n o d n i k i l b u p e a r i s g e n ...
4,5,k pdt sus phi g e n n demi keadilan berdasarka...,2022.0,Perselisihan Hubungan Industrial,NaN,an atas putusan dalam p k perkara ini uitvoerb...,Dikabulkan,a i s e n o d n i k i l b u p e a r i s g e n ...


In [4]:
documents=df["isi_putusan"].fillna("")

In [5]:
vectorizer=TfidfVectorizer(
    stop_words=None,
    max_features=5000
)

tfidf_matrix=vectorizer.fit_transform(documents)

print(tfidf_matrix.shape)

(30, 5000)


In [6]:
similarity_matrix=cosine_similarity(tfidf_matrix)

print(similarity_matrix.shape)

(30, 30)


In [7]:
similarity_df=pd.DataFrame(
    similarity_matrix
)

similarity_df.head()

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,1.000000,0.705508,0.769087,0.797453,0.803908,0.826410,0.739372,0.781273,0.792650,0.727926,...,0.733071,0.741885,0.419819,0.802240,0.796165,0.733290,0.788954,0.825983,0.814237,0.743465
1,0.705508,1.000000,0.674086,0.706251,0.704567,0.721179,0.796303,0.644130,0.662643,0.595670,...,0.628176,0.656696,0.285655,0.708940,0.691768,0.603504,0.687235,0.729569,0.701371,0.667765
2,0.769087,0.674086,1.000000,0.776739,0.784447,0.803541,0.721453,0.744787,0.753851,0.678723,...,0.698712,0.715564,0.404578,0.764327,0.764420,0.692130,0.745104,0.774011,0.785258,0.713632
3,0.797453,0.706251,0.776739,1.000000,0.831279,0.835466,0.758157,0.777474,0.801645,0.743648,...,0.758819,0.767367,0.368527,0.813373,0.830839,0.748408,0.805387,0.823891,0.831320,0.752618
4,0.803908,0.704567,0.784447,0.831279,1.000000,0.843276,0.766892,0.805742,0.805331,0.723929,...,0.752405,0.780189,0.381259,0.820603,0.804429,0.739916,0.792003,0.839346,0.839313,0.750223


In [8]:
def retrieve_cases(case_index,k=5):

    similarity_scores=list(
        enumerate(
            similarity_matrix[case_index]
        )
    )

    similarity_scores=sorted(
        similarity_scores,
        key=lambda x:x[1],
        reverse=True
    )

    similarity_scores=similarity_scores[1:k+1]

    return similarity_scores

In [9]:
retrieve_cases(0)

[(5, np.float64(0.8264101624467864)),
 (27, np.float64(0.8259825378244929)),
 (28, np.float64(0.8142369528507195)),
 (11, np.float64(0.8111161961215274)),
 (13, np.float64(0.8058832599311097))]

In [10]:
top_cases=retrieve_cases(0)

for idx,score in top_cases:

    print("="*80)

    print("Similarity :",round(score,4))

    print(df.iloc[idx]["nomor_perkara"])

    print(df.iloc[idx]["amar_putusan"])

Similarity : 0.8264
k pdt sus phi g e demi keadilan berdasarkan ketuhanan yang maha esa n n m a h k a m a h a g u n g memueriksa perkara perdata khusus perselisihan hubungan industriaol pada tingkat kasasi telah memutus sebagai berikut dalam perkara antarda g surianto bertempat tinggal di dusun vi kelurahan desa n a tembung kecamatan percut sei tuan kabupaten deli i serdang dalam hal ini memberikan kua sa kepada hotman h k manullang s h dan kawan para advokat dan konsultan a i hukum pada kantor hotman partners beralamat di jalan l m sakura iii nomor bkelurahan tanjung selamat kecamatan medan tuuntungan provinsi sumatera utara a berdasarkan surat kuasa khusus tanggal april p k pemohon kasasi e h l a w a n a r a pt syukur indah mulia le polonia hotel i s m conv ention medan berkedudukan di jalan jendral g e sudirman nomor kelurahan madras hulu kecamatan n nmedan polonia kota medan yang diwakili oleh suparjo usebagai direktur dalam hal ini memberikan kuasa kepoada nurmahadi darmawan s h d

In [11]:
retrieval_results=[]

for i in range(len(df)):

    top5=retrieve_cases(i)

    for idx,score in top5:

        retrieval_results.append({

            "query_case":i,

            "retrieved_case":idx,

            "similarity":score

        })

retrieval_df=pd.DataFrame(
    retrieval_results
)

retrieval_df.head()

,query_case,retrieved_case,similarity
0,0,5,0.826410
1,0,27,0.825983
2,0,28,0.814237
3,0,11,0.811116
4,0,13,0.805883


In [12]:
output=os.path.join(
    PROJECT_FOLDER,
    "data",
    "results",
    "retrieval_results.csv"
)

retrieval_df.to_csv(
    output,
    index=False
)

print(output)

/content/drive/MyDrive/CBR-PHI/data/results/retrieval_results.csv


In [13]:
retrieval_df.head()

,query_case,retrieved_case,similarity
0,0,5,0.826410
1,0,27,0.825983
2,0,28,0.814237
3,0,11,0.811116
4,0,13,0.805883


In [14]:
retrieval_df.shape

(150, 3)